# Stage 15 — Energy and AI revision experiments (remote CPU + A100)

All scientific computation runs on **Colab's remote CPU/GPU**. This delivery is
prepared on your device using file I/O and syntax checks only. No scientific
analyses or model fits have been executed locally.

This notebook completes the planned E1–E6 package:

- **E1:** recompute D-BSI on development telemetry; equal weights, single
  components, component removal, threshold/capacity perturbations, 100 bounded
  weight draws, and training-only nonnegative residual weighting.
- **E2:** horizon, anonymous manufacturer and joint subgroups; matched vehicle
  bootstrap intervals, cluster sign-flip tests, within-family Holm correction,
  vehicle-macro MAE and saved-interval coverage.
- **E3:** exact Nissan health-target lineage and shared-checkpoint clustering;
  label plateaus/increases; optional bounded original EVBattery label sample;
  hypothetical label-noise sensitivity; actual logged physics/driver-ID losses.
- **E4:** genuinely future responses at 30/60/120 seconds from the last 120
  seconds, using causal joins and backward derivatives. RWTH leave-one-vehicle-out
  and EVE whole-session chronological splits. Matched persistence,
  constrained longitudinal/electrothermal baseline, physics-feature Ridge,
  context HGB, context+driver HGB and dual-stream
  concept models with KAN or MLP. Three seeds; up to 30 epochs; patience 6.
- **E5:** reuse original fleet KAN checkpoints; choose each switching threshold
  on the original outer-training inner validation vehicles, then evaluate on
  that fold's held-out vehicles.
- **E6:** saved fleet KAN/MLP/TSLANet CPU latency, sizes and sampled process memory;
  concept perturbation stability across three seeds. Cloud hardware is recorded.

E7 needs a new multi-vehicle longitudinal driver+health dataset and is not
included. New analyses are post-hoc strengthening experiments on known datasets;
the frozen Nissan locked-test model stays fixed. Nissan locked-test **saved
predictions and metadata** are audited, without reopening its tensor or retraining.

**Estimated first A100 execution: 2–4 hours**, with an allowance of up to 6 hours
for EVE downloads and remote preparation. Actual time is unknown until this
new protocol runs. Previously completed benchmark training is reused.

Use Drive for this package (>50 MB), or choose browser upload if preferred.
The final block downloads a ZIP with every result, model, history, audit, source,
test log and figure, and saves a copy in Drive when enabled.

In [ ]:
# BLOCK 1 — Fixed protocol and storage choices
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
INPUT_MODE = 'drive'  # 'drive' or 'upload'
DRIVE_PACKAGE = '/content/drive/MyDrive/DBPatchKAN_stage15/DBPatchKAN_Stage15_Energy_AI_Review_Upload.zip'
DRIVE_RUNS_DIR = '/content/drive/MyDrive/DBPatchKAN_stage15/runs'
RAW_LABEL_DIR = '/content/drive/MyDrive/DBPatchKAN_external'
SAVE_TO_DRIVE = True
RUN_ID = 'stage15_energy_ai_review_v1'
SEEDS = [42, 52, 62]
RESPONSE_EPOCHS = 30
BOOTSTRAP_DRAWS = 10000
MAX_RWTH_EXAMPLES_PER_VEHICLE = 1500
DOWNLOAD_EVE = True
RUN_ORIGINAL_LABEL_SAMPLE = True
MAX_ORIGINAL_LABEL_MEMBERS_PER_ARCHIVE = 10000
INPUT_ROOT = Path('/content/dbpatchkan_stage15_input')
PROJECT_ROOT = Path('/content/dbpatchkan_stage15_project')
def update(message):
    print(f'[{datetime.now(timezone.utc).strftime("%H:%M:%S")} UTC] {message}', flush=True)
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
update('BLOCK 1 complete. Computation will run on the remote Colab machine.')

In [ ]:
# BLOCK 2 — Acquire the single package; duplicate upload names are supported
if INPUT_MODE == 'drive' or SAVE_TO_DRIVE or RUN_ORIGINAL_LABEL_SAMPLE:
    from google.colab import drive
    drive.mount('/content/drive')
if INPUT_MODE == 'drive':
    package = Path(DRIVE_PACKAGE)
    assert package.exists(), f'Upload the ZIP to this Drive path first: {package}'
else:
    from google.colab import files
    update('Upload DBPatchKAN_Stage15_Energy_AI_Review_Upload.zip')
    uploaded=files.upload()
    zip_items=[(name,value) for name,value in uploaded.items() if name.lower().endswith('.zip')]
    assert len(zip_items)==1, 'Upload exactly one complete ZIP'
    package=Path('/content/stage15_browser_upload.zip')
    package.write_bytes(zip_items[0][1])
def extract_safely(archive, root):
    for entry in archive.infolist():
        normalized=entry.filename.replace('\\','/')
        assert not normalized.startswith('/') and ':' not in normalized.split('/')[0]
        destination=(root/normalized).resolve()
        assert destination==root.resolve() or root.resolve() in destination.parents, entry.filename
        assert (entry.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink ZIP member refused'
    archive.extractall(root)
update(f'BLOCK 2 extracting {package.stat().st_size/2**20:.1f} MB on remote storage')
with zipfile.ZipFile(package) as archive: extract_safely(archive,INPUT_ROOT)
manifest=json.loads((INPUT_ROOT/'STAGE15_UPLOAD_MANIFEST.json').read_text())
PROJECT=INPUT_ROOT/'project'
assert (PROJECT/'src/dbpatchkan/analysis/stage15_review.py').exists()
assert manifest['locked_test_tensor_included'] is False
RUN_ROOT=(Path(DRIVE_RUNS_DIR)/RUN_ID) if SAVE_TO_DRIVE else (Path('/content/dbpatchkan_stage15_runs')/RUN_ID)
RUN_ROOT.mkdir(parents=True,exist_ok=True)
RESULTS=RUN_ROOT/'results';RESULTS.mkdir(exist_ok=True)
PREPARED=RUN_ROOT/'prepared'
update(f'BLOCK 2 complete: inputs={INPUT_ROOT}, outputs={RUN_ROOT}')

In [ ]:
# BLOCK 3 — Remote installation, immutable inputs and runtime verification
update('BLOCK 3 installing the project on Colab')
subprocess.run([sys.executable,'-m','pip','install','-q','numpy>=2.0','pandas>=2.2','pyarrow>=16',
                'scikit-learn>=1.5','scipy','matplotlib','psutil','pytest'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(PROJECT)],check=True)
source_root=str(PROJECT/'src')
sys.path.insert(0,source_root)
os.environ['PYTHONPATH']=source_root+os.pathsep+os.environ.get('PYTHONPATH','')
os.environ['OMP_NUM_THREADS']='2';os.environ['MKL_NUM_THREADS']='2'
import importlib
for module_name in list(sys.modules):
    if module_name=='dbpatchkan' or module_name.startswith('dbpatchkan.'):del sys.modules[module_name]
importlib.invalidate_caches()
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(2**20),b''):h.update(chunk)
    return h.hexdigest()
for i,(name,record) in enumerate(manifest['files'].items(),1):
    path=INPUT_ROOT/name
    assert path.exists() and path.stat().st_size==record['bytes'],f'Missing/incomplete {name}'
    assert sha256(path)==record['sha256'],f'Hash mismatch: {name}'
    if i==1 or i%10==0:update(f'Input integrity: {i}/{len(manifest["files"])} files verified')
import torch, numpy as np, pandas as pd, dbpatchkan
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > A100 GPU'
update(f'GPU={torch.cuda.get_device_name(0)}; Python={sys.version.split()[0]}; torch={torch.__version__}')
if 'A100' not in torch.cuda.get_device_name(0):update('A different GPU is active; runtime may be longer.')
protocol={'run_id':RUN_ID,'seeds':SEEDS,'response_epochs':RESPONSE_EPOCHS,
    'bootstrap_draws':BOOTSTRAP_DRAWS,'rwth_cap':MAX_RWTH_EXAMPLES_PER_VEHICLE,'eve':DOWNLOAD_EVE,
    'raw_label_sample':RUN_ORIGINAL_LABEL_SAMPLE,'raw_label_members':MAX_ORIGINAL_LABEL_MEMBERS_PER_ARCHIVE,
    'upload_manifest_sha256':sha256(INPUT_ROOT/'STAGE15_UPLOAD_MANIFEST.json'),
    'locked_test_tensor_opened':False,'source_sha256':sha256(PROJECT/'src/dbpatchkan/analysis/stage15_response.py')}
protocol_path=RUN_ROOT/'STAGE15_RUN_PROTOCOL.json'
if protocol_path.exists():assert json.loads(protocol_path.read_text())==protocol,'Resume settings differ; use a new RUN_ID.'
else:protocol_path.write_text(json.dumps(protocol,indent=2))
def run(command,label):
    update(f'START {label}');started=time.time();log=RUN_ROOT/(label.replace(" ","_")+'.log')
    # Stream output from the actual worker, with an elapsed-time heartbeat for long I/O.
    import queue, threading
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy())
    messages=queue.Queue()
    def reader():
        for line in process.stdout:messages.put(line)
        messages.put(None)
    thread=threading.Thread(target=reader,daemon=True);thread.start()
    with log.open('a',encoding='utf-8') as handle:
        while True:
            try:line=messages.get(timeout=30)
            except queue.Empty:
                update(f'{label}: elapsed={(time.time()-started)/60:.1f} min; worker active');continue
            if line is None:break
            print(line,end='',flush=True);handle.write(line);handle.flush()
    rc=process.wait()
    assert rc==0,f'{label} failed with code {rc}; log={log}. Fix and rerun this block.'
    update(f'DONE {label}: {(time.time()-started)/60:.1f} min')
CPU=[sys.executable,'-u','-m','dbpatchkan.analysis.stage15_review','--input-dir',str(INPUT_ROOT),
     '--output-dir',str(RESULTS),'--bootstrap-draws',str(BOOTSTRAP_DRAWS)]
RESPONSE=[sys.executable,'-u','-m','dbpatchkan.analysis.stage15_response','--input-dir',str(INPUT_ROOT),
          '--output-dir',str(RESULTS),'--prepared-dir',str(PREPARED)]
update('BLOCK 3 complete; source path is explicitly active in notebook and worker imports')

In [ ]:
# BLOCK 4 — Remote scientific integrity tests (no local execution)
run([sys.executable,'-m','pytest','-q',str(PROJECT/'tests/test_stage15_remote.py')], 'remote_integrity_tests')
update('BLOCK 4 complete: boundary invariance, missing temperature, cluster units and neural shapes verified')

In [ ]:
# BLOCK 5 — E3 critical label lineage and actual-loss audit
command=CPU+['--tasks','e3']
raw_path=Path(RAW_LABEL_DIR)
if RUN_ORIGINAL_LABEL_SAMPLE and raw_path.exists():
    command+=['--raw-label-dir',str(raw_path),'--raw-label-members',str(MAX_ORIGINAL_LABEL_MEMBERS_PER_ARCHIVE)]
elif RUN_ORIGINAL_LABEL_SAMPLE:
    update('Original archives were not found in Drive. The report will identify this measurement-noise limitation.')
run(command,'E3_label_and_loss_audit')
display(pd.read_csv(RESULTS/'E3_nissan_independent_counts.csv'))
display(json.loads((RESULTS/'E3_auxiliary_driver_audit.json').read_text()))
update('BLOCK 5 complete: saved locked predictions audited; no locked tensor opened')

In [ ]:
# BLOCK 6 — E1 D-BSI sensitivity and development-only downstream regressions
if not (RESULTS/'E1_COMPLETE.json').exists():run(CPU+['--tasks','e1'],'E1_DBSI_robustness')
else:update('E1 already complete: reusing saved analysis')
display(pd.read_csv(RESULTS/'E1_rank_distribution_robustness.csv').head(20))
update('BLOCK 6 complete')

In [ ]:
# BLOCK 7 — E2 horizon/manufacturer statistics from frozen predictions
if not (RESULTS/'E2_COMPLETE.json').exists():run(CPU+['--tasks','e2'],'E2_subgroup_statistics')
else:update('E2 already complete: reusing saved analysis')
comparison=pd.read_csv(RESULTS/'E2_cluster_comparisons.csv')
display(comparison[comparison.candidate.eq('PersistenceAnchored_KAN')])
update('BLOCK 7 complete')

In [ ]:
# BLOCK 8 — E4 causal native telemetry preparation on the remote CPU
command=RESPONSE+['--task','prepare','--max-per-vehicle',str(MAX_RWTH_EXAMPLES_PER_VEHICLE)]
if not DOWNLOAD_EVE:command+=['--skip-eve']
audit_path=PREPARED/'E4_PREPARATION_AUDIT.json'
ready=audit_path.exists() and (PREPARED/'rwth_future_response.npz').exists()
if ready and DOWNLOAD_EVE:ready=(PREPARED/'eve_future_response.npz').exists()
if not ready:run(command,'E4_native_causal_preparation')
else:update('E4 native preparation already available; reusing cached tensors')
display(json.loads(audit_path.read_text()))
update('BLOCK 8 complete; gaps are not expanded and future samples are excluded from inputs')

In [ ]:
# BLOCK 9 — E4 matched future-response training on A100
command=RESPONSE+['--task','train','--seeds',*[str(x) for x in SEEDS],'--epochs',str(RESPONSE_EPOCHS)]
run(command,'E4_matched_future_response_training')
display(pd.read_csv(RESULTS/'E4_matched_future_response_metrics.csv'))
update('BLOCK 9 complete')

In [ ]:
# BLOCK 10 — E5 selective persistence using the original inner validation sets
if not (RESULTS/'E5_COMPLETE.json').exists():run(CPU+['--tasks','e5'],'E5_inner_selected_persistence_switch')
else:update('E5 already complete: reusing saved thresholds and predictions')
display(pd.read_csv(RESULTS/'E5_inner_selected_thresholds.csv'))
display(pd.read_csv(RESULTS/'E5_comparison_metrics.csv'))
update('BLOCK 10 complete: no threshold was chosen from E2 held-out subgroup results')

In [ ]:
# BLOCK 11 — E6 cloud CPU efficiency and named-concept response stability
run(RESPONSE+['--task','benchmark'],'E6_remote_CPU_efficiency')
display(pd.read_csv(RESULTS/'E6_remote_CPU_latency.csv'))
run(CPU+['--tasks','figures'],'review_figures')
update('BLOCK 11 complete; measured latency belongs to the Colab CPU')

In [ ]:
# BLOCK 12 — Manuscript revision evidence and completeness audit
needed={'E1':'E1_COMPLETE.json','E2':'E2_COMPLETE.json','E3':'E3_COMPLETE.json',
        'E4':'E4_TRAIN_COMPLETE.json','E5':'E5_COMPLETE.json','E6':'E6_remote_CPU_latency.csv'}
status={key:(RESULTS/file).exists() for key,file in needed.items()}
pending=[]
prep=json.loads((PREPARED/'E4_PREPARATION_AUDIT.json').read_text()) if (PREPARED/'E4_PREPARATION_AUDIT.json').exists() else {}
if DOWNLOAD_EVE and prep.get('status',{}).get('EVE')!='prepared':pending.append('EVE native response experiment: data preparation/download incomplete')
scope=json.loads((RESULTS/'E3_SCOPE.json').read_text()) if (RESULTS/'E3_SCOPE.json').exists() else {}
if scope.get('original_label_status','').startswith('unavailable'):pending.append('Original EVBattery repeated-label sample unavailable; hypothetical noise sensitivity is not measurement calibration')
pending.extend(['E7: new multi-vehicle longitudinal driving+health data needed for a general driver-induced degradation claim',
                'Physical stress thresholds remain engineering assumptions; sensitivity does not calibrate pack aging',
                'CPU energy use was not measured with a power meter',
                'Manuscript conclusions require inspection of the returned experiment results'])
audit={'completed':status,'pending_or_scope_limits':pending,'locked_tensor_used':False,
       'local_scientific_computation_used':False,'remote_runtime':True,
       'gpu':torch.cuda.get_device_name(0),'completed_utc':datetime.now(timezone.utc).isoformat(),
       'prior_predictions_sha256':sha256(INPUT_ROOT/'fleet_predictions.csv')}
(RUN_ROOT/'STAGE15_COMPLETENESS.json').write_text(json.dumps(audit,indent=2))
inventory=[{'path':str(p.relative_to(RUN_ROOT)),'bytes':p.stat().st_size} for p in RUN_ROOT.rglob('*') if p.is_file()]
pd.DataFrame(inventory).to_csv(RUN_ROOT/'artifact_inventory.csv',index=False)
lines=['# Stage 15: remote revision experiments','',f'Completion: {status}','',
       '## Evidence to incorporate into the manuscript','',
       '- E1: sensitivity/rank stability and development-only downstream value; no universal calibration claim.',
       '- E2: paired vehicle uncertainty, horizon/manufacturer heterogeneity and actual saved interval coverage.',
       '- E3: independent health transition/destination counts, label variation and active logged training terms.',
       '- E4: fixed-horizon future responses on native telemetry; distinguish measured temperature from missing RWTH thermal labels.',
       '- E5: per-fold inner-selected deployment switch; report performance even when persistence is selected.',
       '- E6: real checkpoint sizes, same-hardware CPU latency, memory and concept perturbation stability.',
       '', '## Remaining limits','']+['- '+x for x in pending]
(RUN_ROOT/'STAGE15_MANUSCRIPT_EVIDENCE.md').write_text('\n'.join(lines))
display(audit)
update('BLOCK 12 complete. Inspect metrics and limits before revising superiority or causal claims.')

In [ ]:
# BLOCK 13 — Package and automatically download all results and model checkpoints
# This block can also be run after an interrupted stage to recover partial work.
from google.colab import files
shutil.copy2(INPUT_ROOT/'STAGE15_UPLOAD_MANIFEST.json',RUN_ROOT/'STAGE15_UPLOAD_MANIFEST.json')
for folder in ('fleet_checkpoints','benchmark_checkpoints'):
    shutil.copytree(INPUT_ROOT/folder,RUN_ROOT/'reused_model_checkpoints'/folder,dirs_exist_ok=True)
source_archive=RUN_ROOT/'EXECUTED_SOURCE.zip'
with zipfile.ZipFile(source_archive,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for path in PROJECT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.suffix in {'.py','.toml','.md','.ipynb'}:
            archive.write(path,path.relative_to(PROJECT))
archive_path=Path('/content/DBPatchKAN_stage15_ENERGY_AI_REVIEW_ALL_MODELS_RESULTS.zip')
update('BLOCK 13 creating the final model/result ZIP on Colab')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for path in RUN_ROOT.rglob('*'):
        if not path.is_file() or 'eve_raw' in path.parts or path.name=='latest.pt':continue
        # Keep final model weights and all results; resume/optimizer state stays in the run folder.
        archive.write(path,path.relative_to(RUN_ROOT))
if SAVE_TO_DRIVE:
    destination=Path(DRIVE_RUNS_DIR)/archive_path.name
    shutil.copy2(archive_path,destination)
    update(f'Drive backup saved: {destination}')
update(f'Final archive: {archive_path.stat().st_size/2**20:.1f} MB')
files.download(str(archive_path))
update('BLOCK 13 complete: download triggered. Check the ZIP/Drive backup before stopping the session.')